# Midterm Exam - Model Comparison - Matthew Shaver

This notebook compares the saved Model A classifier with the saved SRGAN pipeline. Model A is tested on the original 128x128 images and on SRGAN-generated 128x128 images from the same untouched test split.

## Setup

Load the libraries, folders, and the two saved models. No model is trained in this notebook.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, auc, f1_score, roc_curve
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import mobilenet_v2

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'notebooks').exists():
    raise FileNotFoundError('Open this notebook from the project repository.')

print('PyTorch version:', torch.__version__)
print('Project root:', PROJECT_ROOT)

In [ ]:
LOW_RESOLUTION_SIZE = 32
IMAGE_SIZE = 128
BATCH_SIZE = 32
CLASS_NAMES = ('cats', 'dogs')

SPLIT_DIR = PROJECT_ROOT / 'data' / 'splits'
MODELS_DIR = PROJECT_ROOT / 'models'
MODEL_A_PATH = MODELS_DIR / 'model_a_mobilenetv2_best.pt'
SRGAN_PATH = MODELS_DIR / 'srgan' / 'last.pt'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'model_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'

if not MODEL_A_PATH.exists() or not SRGAN_PATH.exists():
    raise FileNotFoundError('Run notebooks 02 and 03 before this notebook.')

print('Device:', DEVICE)
print('Model A:', MODEL_A_PATH)
print('SRGAN:', SRGAN_PATH)

## Test Data

Each test image is prepared two ways: as the normal 128x128 input for Model A and as a 32x32 input for the SRGAN.

In [ ]:
test_table = pd.read_csv(SPLIT_DIR / 'test_split.csv')
print('Test images:', len(test_table))
print(test_table['label'].value_counts().sort_index())

normal_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

tensor_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [ ]:
class ComparisonDataset(Dataset):
    def __init__(self, records):
        self.records = records.reset_index(drop=True)
        self.class_to_index = {'cats': 0, 'dogs': 1}

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records.iloc[index]
        with Image.open(record['image_path']) as image:
            image = image.convert('RGB')

        original_image = normal_transform(image)
        high_resolution = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BICUBIC)
        low_resolution = high_resolution.resize((LOW_RESOLUTION_SIZE, LOW_RESOLUTION_SIZE), Image.BICUBIC)
        label = self.class_to_index[record['label']]
        return tensor_transform(low_resolution), original_image, torch.tensor(label, dtype=torch.float32)


test_loader = DataLoader(
    ComparisonDataset(test_table),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=PIN_MEMORY,
)

print('Test batches:', len(test_loader))

## Load Saved Models

The classifier architecture matches notebook 02. The generator architecture matches notebook 03.

In [ ]:
class MobileNetV2Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = mobilenet_v2(weights=None)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, 1)

    def forward(self, images):
        return self.model(images).squeeze(1)


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.PReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
        )

    def forward(self, images):
        return images + self.layers(images)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.initial = nn.Sequential(nn.Conv2d(3, 64, kernel_size=9, padding=4), nn.PReLU())
        self.residuals = nn.Sequential(*[ResidualBlock(64) for _ in range(4)])
        self.after_residuals = nn.Sequential(nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64))
        self.upsample = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=3, padding=1), nn.PixelShuffle(2), nn.PReLU(),
            nn.Conv2d(64, 256, kernel_size=3, padding=1), nn.PixelShuffle(2), nn.PReLU(),
        )
        self.output = nn.Sequential(nn.Conv2d(64, 3, kernel_size=9, padding=4), nn.Tanh())

    def forward(self, images):
        initial_features = self.initial(images)
        features = self.after_residuals(self.residuals(initial_features)) + initial_features
        return self.output(self.upsample(features))

In [ ]:
model_a = MobileNetV2Classifier().to(DEVICE)
model_a.load_state_dict(torch.load(MODEL_A_PATH, map_location=DEVICE))
model_a.eval()

generator = Generator().to(DEVICE)
checkpoint = torch.load(SRGAN_PATH, map_location=DEVICE)
generator.load_state_dict(checkpoint['generator_state_dict'])
generator.eval()

print('Loaded Model A')
print('Loaded SRGAN checkpoint from epoch:', checkpoint['epoch'])

## Compare Performance

The same saved Model A makes both sets of predictions. The first uses original images; the second uses images produced by the saved SRGAN. This shows how the SRGAN image-generation step affects classification performance.

In [ ]:
original_probabilities = []
srgan_probabilities = []
targets = []

with torch.no_grad():
    for low_resolution, original_images, labels in test_loader:
        low_resolution = low_resolution.to(DEVICE, non_blocking=True)
        original_images = original_images.to(DEVICE, non_blocking=True)
        generated_images = generator(low_resolution)

        original_probabilities.extend(torch.sigmoid(model_a(original_images)).cpu().tolist())
        srgan_probabilities.extend(torch.sigmoid(model_a(generated_images)).cpu().tolist())
        targets.extend(labels.int().tolist())

def get_metrics(probabilities, targets):
    predictions = [int(probability >= 0.5) for probability in probabilities]
    false_positive_rate, true_positive_rate, _ = roc_curve(targets, probabilities)
    return {
        'Accuracy': accuracy_score(targets, predictions),
        'F1': f1_score(targets, predictions, zero_division=0),
        'AUC': auc(false_positive_rate, true_positive_rate),
    }, false_positive_rate, true_positive_rate

original_metrics, original_fpr, original_tpr = get_metrics(original_probabilities, targets)
srgan_metrics, srgan_fpr, srgan_tpr = get_metrics(srgan_probabilities, targets)

comparison_table = pd.DataFrame(
    [original_metrics, srgan_metrics],
    index=['Model A: original images', 'Model A + SRGAN images'],
)
comparison_table.round(4)

In [ ]:
comparison_table.to_csv(OUTPUT_DIR / 'model_comparison_metrics.csv')
(OUTPUT_DIR / 'model_comparison_metrics.json').write_text(
    json.dumps(comparison_table.to_dict(), indent=2),
    encoding='utf-8',
)

plt.figure(figsize=(7, 5))
plt.plot(original_fpr, original_tpr, label=f"Original images (AUC = {original_metrics['AUC']:.3f})")
plt.plot(srgan_fpr, srgan_tpr, label=f"SRGAN images (AUC = {srgan_metrics['AUC']:.3f})")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_comparison.png', dpi=200)
plt.show()

print('Comparison results saved to:', OUTPUT_DIR)